## Run InterProScan

In [1]:
!curl -O http://ftp.ebi.ac.uk/pub/software/unix/iprscan/5/5.77-108.0/alt/interproscan-data-5.77-108.0.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6580M  100 6580M    0     0  2960k      0  0:37:56  0:37:56 --:--:-- 1713k


In [1]:
!cp ../Bradysia_coprophila/endosymbiont_analysis.ipynb ./

In [7]:
!tar -pxzf interproscan-data-5.77-108.0.tar.gz
!singularity pull docker://interpro/interproscan:5.77-108.0

In [3]:
!mkdir /net/bos-nfsisilon/ifs/rc_labs/extavour_lab/rkapoor/Bradysia_HGT_2026/outputs/interproscan

In [1]:
!sbatch scripts/interproscan.sh

## Tabulate HTGS/domain

In [1]:
import pandas as pd
df=pd.read_csv("outputs/interproscan/longest_isoform.fasta.tsv",sep="\t",header=None)

In [19]:
dfriso=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col="accession")
hgts=set(dfriso[dfriso.tree_HTG].index)

In [116]:
df.columns=[
    "Protein accession",
    "Sequence MD5 digest",
    "Sequence length",
    "Analysis",
    "Signature accession",
    "Signature description",
    "Start location",
    "Stop location",
    "Score",
    "Status",
    "Date",
    "InterPro accession",
    "InterPro description",
    "GO annotations",
    "Pathways annotations"
]

In [117]:
df=df[df['InterPro description']!='-']

In [118]:
df=df[df['Protein accession'].isin(hgts)]


In [119]:
idf=pd.DataFrame()
for p, dfi in df.groupby('InterPro description'):
    i=dfi['InterPro accession'].values[0]
    n=f'{p} ({i})'
    idf.loc[n,'n_HTGs']=len(set(dfi['Protein accession']))
    idf.loc[n,'HTGs']=str(set(dfi['Protein accession']))

In [120]:
idf.sort_values('n_HTGs',ascending=False).to_csv("outputs/interproscan_htgs_counts.tsv",sep="\t")

In [9]:
idf.to_csv("outputs/interproscan_htgs_counts.tsv",sep="\t")

In [1]:
import pandas as pd
idf=pd.read_csv("outputs/interproscan_htgs_counts.tsv",sep="\t",index_col=0)

## Retrieve InterPro domain descriptions

In [3]:
import requests
import bs4
from bs4 import BeautifulSoup

def get_interpro_description(ipr_id):
    url = f"https://www.ebi.ac.uk/interpro/api/entry/interpro/{ipr_id}/"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    # Inspect once if the field name differs by API version:
    # print(data.keys())
    # print(data.get("metadata", {}).keys())

    md = data.get("metadata", {})
    desc = md.get("description") or data.get("description")
    
    if desc is None:
        return None

    # Some InterPro descriptions may contain HTML markup.
    if isinstance(desc, list):
        desc = "\n\n".join(str(x) for x in desc)

    return BeautifulSoup(str(desc), "html.parser").get_text(" ", strip=True)
for index, row in idf.iterrows():
    try:
        i="IPR"+index.split("(IPR")[1].split(")")[0]
        idf.loc[index, 'description']=get_interpro_description(i)
    except:
        print(index)

In [27]:
import ast
for index, row in idf.iterrows():
    if str(row.description)!='nan':
        idf.loc[index,'description']=str(row.description).split(": ")[1].split("', 'llm")[0].replace("' ","").replace("\r\n\r\n","")


In [35]:
for index, row in idfi.iterrows():
    if str(row.description)!='nan':
        idf.loc[index,'description']=str(row.description).replace("\t","").strip()

In [189]:
df=pd.read_csv("outputs/curated_interproscan_htgs_counts_edited.csv",index_col=0)

In [190]:
import ast
df['HTGs']=[ast.literal_eval(x) for x in df.HTGs ]
df=df.explode('HTGs')

In [191]:
dfp=pd.DataFrame()
for h, dfi in df.groupby('HTGs'):
    dfp.loc[h,'Pathway(s)']=str(set(dfi['Pathway']))

In [ ]:
dfp=pd.DataFrame()
for h, dfi in df.groupby('HTGs'):
    dfp.loc[h,'Pathway(s)']=str(set(dfi['Pathway']))
    dfp.loc[h,'Sources(s)']=str(set(dfi['Source']))

In [ ]:
dfp.to_csv("outputs/per_protein_manual_annotation.tsv")

## Load manual sheet after manual annotation
Fill in ancestry, localization, RefSeq annotation

In [3]:
dfp=pd.read_csv("outputs/per_protein_manual_annotation.tsv",sep="\t",index_col=0)

In [6]:
for index, row in dfp.iterrows():
    dfp.loc[index,['RefSeq description (v2)','DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[index,['description','domain','kingdom','phylum','class','order']].values

In [7]:
from Bio import SeqIO
input_file = open(f"datasets/GCF_014529535.1_BU_Bcop_v1_protein.faa")
old_fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
for index, row in dfp.iterrows():
    if index in old_fasta:
        dfp.loc[index,'RefSeq description (v1)']=" ".join(old_fasta[index].description.split(" ")[1:])

In [9]:
dfs=pd.read_csv('outputs/sister_automated_per_sequence.tsv',sep='\t',index_col='cluster_seq')
for index, row in dfp.iterrows():
    dfp.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values

In [23]:
deep=pd.read_csv("outputs/hgt_deeploc_results.csv",index_col=0)

In [10]:
deep=pd.read_csv("outputs/hgt_deeploc_results.csv",index_col=0)
for index, row in dfp.iterrows():
    dfp.loc[index,'localization']=deep.loc[index,'Localizations']

In [15]:
for index, row in dfp.iterrows():
    d=list(idf[idf.HTGs.str.contains(index)].index)
    dfp.loc[index,'Domains']=";".join(d)

In [19]:
for index, row in dfp.iterrows():

    dfp.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]
    dfp.loc[index,'tree']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].index.values[0]

In [20]:
for i, row in dfp.iterrows():
    if str(row.tree_relative)=='nan':
    
        if row.tree_phylum=='Arthropoda':
            if row.tree_class!='nan':
                dfp.loc[i,'relative']=row.tree_class
        elif row.tree_kingdom=='Metazoa':
            if row.tree_phylum!='nan':
                dfp.loc[i,'relative']=row.tree_phylum
        elif row.tree_kingdom=='Fungi':
            dfp.loc[i,'relative']='Fungi'
        elif row.tree_kingdom=='Viridiplantae':
            dfp.loc[i,'relative']='Viridiplantae'
        elif row.tree_domain=='Eukaryota' and str(row.tree_kingdom)!='nan':
            dfp.loc[i,'relative']='Protist'
        elif row.tree_domain=='Viruses' or row.tree_domain=='Bacteria':
            dfp.loc[i,'relative']=row.tree_domain
        if str(dfp.loc[i,'relative'])=='nan':
            # print(row.DIAMOND_kingdom)
    
            if row.DIAMOND_phylum=='Arthropoda':
                if row.DIAMOND_class!='nan':
                    dfp.loc[i,'relative']=row.DIAMOND_class
            elif row.DIAMOND_kingdom=='Metazoa':
                if row.DIAMOND_phylum!='nan':
                    dfp.loc[i,'relative']=row.DIAMOND_phylum
            elif row.DIAMOND_kingdom=='Fungi':
                dfp.loc[i,'relative']='Fungi'
            elif row['DIAMOND_kingdom']=='Viridiplantae':
                dfp.loc[i,'relative']='Viridiplantae'
            elif row.DIAMOND_domain=='Eukaryota' and row.DIAMOND_kingdom!='nan':
                dfp.loc[i,'relative']='Protist'
            elif row.DIAMOND_domain=='Viruses' or row.DIAMOND_domain=='Bacteria':
                dfp.loc[i,'relative']=row.DIAMOND_domain
    elif str(row.tree_relative)!='nan':

        dfp.loc[i,'relative']=row.tree_relative


In [128]:
dfp.to_csv("outputs/per_protein_manual_annotation.tsv",sep="\t")

In [23]:
dfp.loc[:,['Pathway(s)', 'Manual annotation', 'RefSeq description (v2)',
       'RefSeq description (v1)', 'Domains', 'localization', 'Sources(s)',
       'relative']].to_csv("outputs/per_protein_manual_annotation.tsv",sep="\t")

In [93]:
c="Pathway(s)		Unnamed: 2		RefSeq description (v2)	RefSeq description (v1)	Domains	localization	Sources(s)	DIAMOND_domain	DIAMOND_kingdom	DIAMOND_phylum	DIAMOND_class	DIAMOND_order		tree_domain	tree_kingdom	tree_phylum	tree_class	tree_order"

In [94]:
c=[x for x in c.split("\t") if x!='']

In [128]:
dfp=pd.read_csv("outputs/per_protein_manual_annotation.tsv",sep="\t",index_col=0)
dfp.sort_values(by=['Pathway(s)','Manual annotation']).to_csv("outputs/per_protein_manual_annotation.tsv",sep="\t")

In [96]:
dfp.loc[:,c].to_csv("outputs/per_protein_manual_annotation.tsv",sep="\t")

In [47]:
import pandas as pd
dfp=pd.read_csv('outputs/per_protein_manual_annotation.tsv',sep='\t',index_col=0)

dfp[dfp['Pathway(s)'].str.contains('Proteostasis')&dfp['Manual annotation'].str.contains('chaperone')].shape[0]

11

In [48]:
ps=set()
for x in set(dfp['Pathway(s)']):
    for y in x.split(","):
        ps.add(y.strip())
ps=list(ps)
ps.sort()
dfcount = pd.DataFrame()

for x in ps:
    dfcount.loc[x, 'N_HTGs'] = dfp['Pathway(s)'].str.contains(x, na=False, regex=False).sum()

In [50]:
ps=set()
for x in set(dfp['Pathway(s)']):
    for y in x.split(","):
        ps.add(y.strip().split("(")[0].strip())
ps=list(ps)
ps.sort()
dfcount = pd.DataFrame()

for x in ps:
    dfcount.loc[x, 'N_HTGs'] = dfp['Pathway(s)'].str.contains(x, na=False, regex=False).sum()

In [51]:
dfcount

,N_HTGs
ADP-ribosylation,9.0
Alternative respiratory chain protein,2.0
Amino acid biosynthesis,13.0
Amino acid catabolism,24.0
Antimicrobial/secondary metabolite biosynthesis,56.0
Bacterial cell membrane biosynthesis,9.0
Bacterial cell wall biosynthesis,15.0
Bacterial cell wall degradation/turnover,12.0
Base-excision repair,4.0
Carbohydrate catabolism,119.0


In [17]:
dfp.sort_values(by=['Pathway(s)','Manual annotation']).to_csv('outputs/per_protein_manual_annotation.tsv',sep='\t')

In [63]:
dfp=pd.read_csv('outputs/per_protein_manual_annotation.tsv',sep='\t',index_col=0)

In [67]:

locs=[]
for x in list(dfp[dfp['Pathway(s)'].str.contains('DNA repair')].Localization):
    .extend(x.split("|"))
from collections import Counter
Counter(locs)

Counter({'Nucleus': 14, 'Endoplasmic reticulum': 1, 'Cytoplasm': 5})

In [69]:

locs=[]
for x in list(dfp[dfp['Pathway(s)'].str.contains('Epigenetic regulation')].Localization):
    locs.extend(x.split("|"))
from collections import Counter
Counter(locs)

Counter({'Cytoplasm': 19, 'Nucleus': 24})

In [65]:
locs=[]
for x in list(dfp[dfp['Pathway(s)'].str.contains('Transmembrane transport')].Localization):
    locs.extend(x.split("|"))
from collections import Counter
Counter(locs)

Counter({'Lysosome/Vacuole': 25,
         'Cell membrane': 35,
         'Golgi apparatus': 1,
         'Extracellular': 5,
         'Endoplasmic reticulum': 4,
         'Mitochondrion': 5,
         'Cytoplasm': 3,
         'Nucleus': 2})

## Compare annotations to genes upregulated on irradiation

In [21]:
dnar=set(dfp[dfp['Pathway(s)'].str.contains('DNA repair')].index)
ox=set(dfp[dfp['Pathway(s)'].str.contains('Redox homeostasis/oxidative stress response')].index)
prot=set(dfp[dfp['Pathway(s)'].str.contains('Proteostasis')].index)
xeno=set(dfp[dfp['Pathway(s)'].str.contains('Detoxification/xenobiotic metabolism')].index)

In [26]:
n_htgs=dfriso[dfriso.tree_HTG].shape[0]

In [22]:
dna=pd.read_csv("outputs/irradiation_upregulated_hgts.tsv",sep="\t",index_col=0)
for index, row in dna.iterrows():
    if dfriso[dfriso.gene==index].index[0] in set(dfp.index):
        dna.loc[index,'manual_annotation']=dfp.loc[dfriso[dfriso.gene==index].index[0],'Manual annotation']
    dna.loc[index,'DNA repair']=dfriso[dfriso.gene==index].index[0] in dnar
    dna.loc[index,'Redox homeostasis/oxidative stress response']= dfriso[dfriso.gene==index].index[0] in ox
    dna.loc[index,'Proteostasis']= dfriso[dfriso.gene==index].index[0] in prot
    dna.loc[index,'Detoxification/xenobiotic metabolism']= dfriso[dfriso.gene==index].index[0] in xeno

In [30]:
dna.to_csv("outputs/irradiation_upregulated_hgts.tsv",sep="\t")

In [29]:
n_upreg=dna.shape[0]
n_upreg

40

In [28]:
n_dnar=len(dnar)
n_ox=len(ox)
n_prot=len(prot)
n_xeno=len(xeno)

In [33]:
n_htgs=dfriso[dfriso.tree_HTG].shape[0]

In [30]:
n_upreg_i=dna[dna['DNA repair']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_dnar - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_dnar - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


4 (13.564814814814815, 0.0006011102044663374)


In [31]:
dna[dna['DNA repair']]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,description,DNA repair,Redox homeostasis/oxidative stress response,Proteostasis,manual_annotation,Detoxification/xenobiotic metabolism
LOC119074661,150.694597,1.971377,0.261232,7.546467,4.472234e-14,1.136275e-11,putative 3-methyladenine DNA glycosylase,True,False,False,3-methyladenine DNA glycosylase (MPG),False
LOC119067568,726.345053,1.587958,0.166216,9.553579,1.252920e-21,6.133739e-19,protein argonaute-2-like,True,False,False,Argonaute 2,False
LOC119085358,461.810156,1.785249,0.185331,9.632762,5.814514e-22,2.956009e-19,DNA polymerase kappa-like,True,False,False,DNA polymerase kappa,False
LOC119085384,19443.453743,6.115400,0.141228,43.301673,0.000000e+00,0.000000e+00,poly [ADP-ribose] polymerase tankyrase-like,True,False,False,poly [ADP-ribose] polymerase (PARP) tankyrase-...,False


In [35]:

n_upreg_i=dna[dna['Detoxification/xenobiotic metabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_xeno - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_xeno - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


4 (1.3284600389863548, 0.37871573838100436)


In [33]:

n_upreg_i=dna[dna['Proteostasis']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_xeno - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_xeno - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


7 (2.6104286104286105, 0.03105363308420859)


In [32]:

n_upreg_i=dna[dna['Redox homeostasis/oxidative stress response']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_ox - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_ox - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


3 (1.8198198198198199, 0.25072226111255036)


In [34]:
dna[dna['Redox homeostasis/oxidative stress response']]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,description,DNA repair,Redox homeostasis/oxidative stress response,Proteostasis,manual_annotation,Detoxification/xenobiotic metabolism
LOC119082590,938.885116,0.605664,0.131749,4.597100,4.284118e-06,2.874491e-04,glutathione S-transferase Mu 3-like,False,True,False,Glutathione S-transferase (phase II xenobiotic...,True
LOC119079643,145.982278,0.704714,0.204574,3.444779,5.715263e-04,1.860698e-02,glutathione S-transferase 1-like,False,True,False,Glutathione S-transferase (phase II xenobiotic...,True
LOC119081485,579.385063,1.415988,0.159365,8.885213,6.379854e-19,2.810964e-16,glutathione S-transferase-like,False,True,False,Glutathione S-transferase (phase II xenobiotic...,True


## Compare annotations to genes upregulated in salivary glands

In [25]:
df=pd.read_csv("outputs/carbohydrate_catabolism.tsv",sep="\t",index_col=0)

In [28]:
df.columns

Index(['Final annotation', 'CAZy family', 'Cell wall substrate ', 'source'], dtype='object')

In [36]:
dfp[dfp['Manual annotation'].str.contains('Sodium-dependent glucose transporter')].shape[0]

12

In [31]:
df[df['Cell wall substrate '].astype(str)!='nan'].shape[0]

82

In [49]:
dfp=pd.read_csv("outputs/p_hygida_proteome.tsv",sep='\t',index_col=0)

In [51]:
dfp[dfp['Manual annotation'].str.contains('phase I ')].shape[0]

54

In [52]:
dfp[dfp['Manual annotation'].str.contains('phase II ')].shape[0]

30

In [53]:
Counter(dfp[dfp['Manual annotation'].str.contains('phase II ')]['Manual annotation'])

Counter({'Sulfotransferase (phase II xenobiotic metabolism)': 1,
         'Thiol S-methyltransferase (phase II xenobiotic metabolism)': 6,
         'UDP-glucosyltransferase (phase II xenobiotic metabolism)': 6,
         'Glutathione S-transferase (phase II xenobiotic metabolism)': 17})

In [50]:
Counter(dfp[dfp['Manual annotation'].str.contains('phase I ')]['Manual annotation'])

Counter({'Flavin monooxygenase FMO (phase I xenobiotic metabolism)': 16,
         'Aldo/Keto Reductase (phase I xenobiotic metabolism)': 5,
         'Carbonyl reductase (phase I xenobiotic metabolism)': 1,
         'Cytochrome P450 (phase I xenobiotic metabolism)': 27,
         'Epoxide hydrolase (phase I xenobiotic metabolism)': 1,
         'L-xylulose reductase/carbonyl reductase (phase I xenobiotic metabolism)': 3,
         'Pyridine nucleotide-disulfide oxidoreductase/amino oxidase (phase I xenobiotic metabolism)': 1})

In [54]:
Counter(dfp[dfp['Manual annotation'].str.contains('phase III ')]['Manual annotation'])

Counter({'ATP-binding cassette subfamily G transporter (phase III xenobiotic metabolism)': 1,
         'Multi Antimicrobial Extrusion Protein (phase III xenobiotic metabolism)': 1,
         'Multidrug transporter EmrE superfamily (phase III metabolism)': 1})

In [12]:
carb['Predicted function']=[f"{x} ({y})" for x,y in zip(carb['Predicted function'],carb['CAZy family'])]

In [16]:
import pandas as pd
carb=pd.read_csv("outputs/carbohydrate_catabolism.tsv",sep="\t",index_col=0)
from collections import Counter
f=open("outputs/catabolism_carbs_counts.tsv",'w')
carb['Cell wall substrate ']=carb['Cell wall substrate '].fillna('Other')
carb['Predicted function']=[f"{x} ({y})" for x,y in zip(carb['Predicted function'],carb['CAZy family'])]
for index, dfi in carb.groupby('Cell wall substrate '):
    f.write(f'{index}\t{dfi.shape[0]}\n')
    for i, dfix in dfi.groupby('Predicted function'):
        f.write(f'{i}\t{dfix.shape[0]}\n')
f.close()


In [38]:
carb=set(pd.read_csv("outputs/carbohydrate_catabolism.tsv",sep="\t",index_col=0).index)

In [39]:
prot=set(dfp[dfp['Pathway(s)'].str.contains('Protein catabolism')].index)
aa=set(dfp[dfp['Pathway(s)'].str.contains('Amino acid catabolism')].index)
lipid=set(dfp[dfp['Pathway(s)'].str.contains('Lipid catabolism')].index)

In [40]:
dna=pd.read_csv("outputs/salivary_upregulated_hgts.tsv",sep='\t',index_col=0)

In [41]:
for index, row in dna.iterrows():
    dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
    if dfriso[dfriso.gene==index].index[0] in set(dfp.index):
        dna.loc[index,'manual_annotation']=dfp.loc[dfriso[dfriso.gene==index].index[0],'Manual annotation']
    dna.loc[index,'Carbohydrate catabolism']=dfriso[dfriso.gene==index].index[0] in carb
    dna.loc[index,'Protein catabolism']=dfriso[dfriso.gene==index].index[0] in prot
    dna.loc[index,'Amino acid catabolism']=dfriso[dfriso.gene==index].index[0] in aa
    dna.loc[index,'Lipid catabolism']=dfriso[dfriso.gene==index].index[0] in lipid

/tmp/ipykernel_2803029/3397162442.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
/tmp/ipykernel_2803029/3397162442.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
/tmp/ipykernel_2803029/3397162442.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[

In [42]:
n_upreg=dna.shape[0]
n_prot=len(prot)
n_aa=len(aa)
n_carb=len(carb)
n_lipid=len(lipid)

In [43]:
n_upreg_i=dna[dna['Protein catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_prot - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_prot - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


3 (0.4217948717948718, 0.9679848387116544)


In [44]:
n_upreg_i=dna[dna['Lipid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_lipid - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_lipid - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


7 (1.3654295733489188, 0.2837042344288369)


In [45]:
n_upreg_i=dna[dna['Carbohydrate catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_carb - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_carb - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


5 (0.4498587570621469, 0.9813100926012569)


In [46]:
n_upreg_i=dna[dna['Amino acid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_aa - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_aa - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


1 (0.511177347242921, 0.859097795461893)


In [47]:
dna.to_csv("outputs/salivary_upregulated_hgts.tsv",sep="\t")

## Comparison to larval upregulated genes

In [78]:
dna=pd.read_csv("outputs/larval_upregulated_hgts.tsv",sep="\t",index_col=0)

In [79]:
for index, row in dna.iterrows():
    dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
    if dfriso[dfriso.gene==index].index[0] in set(dfp.index):
        dna.loc[index,'manual_annotation']=dfp.loc[dfriso[dfriso.gene==index].index[0],'Manual annotation']
    dna.loc[index,'Carbohydrate catabolism']=dfriso[dfriso.gene==index].index[0] in carb
    dna.loc[index,'Protein catabolism']=dfriso[dfriso.gene==index].index[0] in prot
    dna.loc[index,'Amino acid catabolism']=dfriso[dfriso.gene==index].index[0] in aa
    dna.loc[index,'Lipid catabolism']=dfriso[dfriso.gene==index].index[0] in lipid
    dna.loc[index,'Detoxification/xenobiotic metabolism']=dfriso[dfriso.gene==index].index[0] in xeno

/tmp/ipykernel_2803029/2920179032.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
/tmp/ipykernel_2803029/2920179032.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[dfriso.gene==index].description[0]
/tmp/ipykernel_2803029/2920179032.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dna.loc[index,'description']=dfriso[

In [80]:
n_upreg=dna.shape[0]
n_prot=len(prot)
n_aa=len(aa)
n_carb=len(carb)
n_lipid=len(lipid)
n_xeno=len(xeno)

In [81]:
n_upreg,n_prot,n_aa,n_carb,n_lipid,n_xeno

(600, 81, 23, 125, 66, 118)

In [82]:
n_upreg_i=dna[dna['Protein catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_prot - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_prot - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


49 (2.459448729582577, 7.371645979856531e-05)


In [83]:
n_upreg_i=dna[dna['Lipid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_lipid - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_lipid - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


36 (1.8872340425531915, 0.008351441815343332)


In [84]:
n_upreg_i=dna[dna['Carbohydrate catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_carb - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_carb - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


79 (2.871109071184178, 2.139420169424026e-08)


In [85]:
n_upreg_i=dna[dna['Amino acid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_aa - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_aa - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


8 (0.8126126126126126, 0.7507244719629445)


In [86]:
n_upreg_i=dna[dna['Detoxification/xenobiotic metabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_xeno - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_xeno - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


49 (1.0929272206002263, 0.3581485093571358)


In [87]:
n_upreg_i=dna[dna['Detoxification/xenobiotic metabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_xeno - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_xeno - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


49 (1.0929272206002263, 0.3581485093571358)


In [58]:
dna.to_csv("outputs/larval_upregulated_hgts.tsv",sep="\t")

## Compare annotations to P. hygida salivary proteome

In [68]:
dna=pd.read_csv("outputs/p_hygida_proteome.tsv",sep="\t",index_col=0)
dna

,Name,prot,B.copro_orthologs,Unique peptides,Carbohydrate catabolism,Protein catabolism,Amino acid catabolism,Lipid catabolism,Detoxification/xenobiotic metabolism
ID,,,,,,,,,
Bhyg_04913-RA,Similar to Endoglucanase (Cryptopygus antarcti...,KAJ6649674.1,"XP_037025249.1, XP_037027558.1, XP_037026636.1...",2,True,False,False,False,False
Bhyg_14441-RA,Similar to Zan Zonadhesin (Mus musculus) AED,KAJ6635855.1,XP_037041478.2,9,False,False,False,False,False
Bhyg_05715-RA,"Similar to MGAM Maltase-glucoamylase, intestin...",KAJ6640783.1,"XP_037045282.1, XP_080113298.1, XP_080115884.1",4,True,False,False,False,False
Bhyg_06428-RA,Similar to Cpn2 Carboxypeptidase N subunit 2 (...,KAJ6641489.1,XP_037041622.1,3,False,False,False,False,False
Bhyg_13495-RA,Similar to Araf43A Extracellular exo-alpha-(1-...,KAJ6634914.1,"XP_037025833.1, XP_037039505.1",2,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...
Bhyg_14474-RA,Protein of unknown function AED,KAJ6635888.1,XP_037037490.1,3,False,False,False,False,False
Bhyg_00919-RA,Similar to acdh-6 Probable acyl-CoA dehydrogen...,KAJ6645711.1,XP_037025034.1,3,False,False,False,True,False
Bhyg_00610-RA,Similar to CHKB Choline/ethanolamine kinase (H...,KAJ6645404.1,"XP_080116649.1, XP_080120124.1, XP_037039380.1...",9,False,False,False,False,False


In [59]:
dna=pd.read_csv("outputs/p_hygida_proteome.tsv",sep="\t",index_col=0)
for index, row in dna.iterrows():
    bcop=set([x.strip() for x in row['B.copro_orthologs'].split(",")])
    dna.loc[index,'Carbohydrate catabolism']=len(carb&bcop)>0
    dna.loc[index,'Protein catabolism']=len(prot&bcop)>0
    dna.loc[index,'Amino acid catabolism']=len(aa&bcop)>0
    dna.loc[index,'Lipid catabolism']=len(lipid&bcop)>0
    dna.loc[index,'Detoxification/xenobiotic metabolism']= len(xeno&bcop)>0

In [60]:
n_xeno=len(xeno)

In [61]:
n_upreg=dna.shape[0]

In [62]:
n_upreg_i=dna[dna['Protein catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_prot - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_prot - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


15 (2.712940212940213, 0.001901003726257338)


In [63]:
n_upreg_i=dna[dna['Carbohydrate catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_carb - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_carb - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


19 (2.152618585787339, 0.005288448423390497)


In [64]:
n_upreg_i=dna[dna['Lipid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_lipid - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_lipid - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


5 (0.9009619292778757, 0.6548194747319217)


In [65]:
n_upreg_i=dna[dna['Amino acid catabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_aa - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_aa - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


2 (1.0522273425499231, 0.5822222299738391)


In [67]:
n_upreg_i=dna[dna['Detoxification/xenobiotic metabolism']].shape[0]
from scipy.stats import fisher_exact, hypergeom

table = [
    [n_upreg_i, n_xeno - n_upreg_i],
    [n_upreg - n_upreg_i, n_htgs - n_xeno - n_upreg + n_upreg_i]
]

print(n_upreg_i,fisher_exact(table, alternative="greater"))


24 (3.246558197747184, 1.3528329936569783e-05)


In [68]:
dna.to_csv("outputs/p_hygida_proteome.tsv",sep="\t")

## Fill in ancestry data for carbohydrate catabolism

In [26]:
dfm=pd.read_csv("outputs/carbohydrate_catabolism.tsv",sep="\t",index_col='HTG')

In [27]:
dfs=pd.read_csv('outputs/sister_automated_per_sequence.tsv',sep='\t',index_col='cluster_seq')

In [28]:
dft=pd.read_csv('outputs/SI_inspection_bradysia_6_23_2026.tsv',sep='\t',index_col=0)

In [29]:
dfriso=pd.read_csv("outputs/per_protein_diamond_data.tsv",sep="\t",index_col="accession")

In [30]:
for m, row in dfm.iterrows():

    dfm.loc[m,['DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[m,['domain','kingdom','phylum','class','order']].values
for index, row in dfm.iterrows():
    dfm.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values
for index, row in dfm.iterrows():

    dfm.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]
    dfm.loc[index,'tree']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].index.values[0]

/tmp/ipykernel_3073646/3653080245.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Collembola' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dfm.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]


In [31]:
for index, row in dfm.iterrows():

    dfm.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]
    dfm.loc[index,'tree']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].index.values[0]

In [35]:
for i, row in dfm.iterrows():
    if str(row.tree_relative)=='nan':
    
        if row.tree_phylum=='Arthropoda':
            if row.tree_class!='nan':
                dfm.loc[i,'relative']=row.tree_class
        elif row.tree_kingdom=='Metazoa':
            if row.tree_phylum!='nan':
                dfm.loc[i,'relative']=row.tree_phylum
        elif row.tree_kingdom=='Fungi':
            dfm.loc[i,'relative']='Fungi'
        elif row.tree_kingdom=='Viridiplantae':
            dfm.loc[i,'relative']='Viridiplantae'
        elif row.tree_domain=='Eukaryota' and str(row.tree_kingdom)!='nan':
            dfm.loc[i,'relative']='Protist'
        elif row.tree_domain=='Viruses' or row.tree_domain=='Bacteria':
            dfm.loc[i,'relative']=row.tree_domain
        if str(dfm.loc[i,'relative'])=='nan':
            # print(row.DIAMOND_kingdom)
    
            if row.DIAMOND_phylum=='Arthropoda':
                if row.DIAMOND_class!='nan':
                    dfm.loc[i,'relative']=row.DIAMOND_class
            elif row.DIAMOND_kingdom=='Metazoa':
                if row.DIAMOND_phylum!='nan':
                    dfm.loc[i,'relative']=row.DIAMOND_phylum
            elif row.DIAMOND_kingdom=='Fungi':
                dfm.loc[i,'relative']='Fungi'
            elif row['DIAMOND_kingdom']=='Viridiplantae':
                dfm.loc[i,'relative']='Viridiplantae'
            elif row.DIAMOND_domain=='Eukaryota' and row.DIAMOND_kingdom!='nan':
                dfm.loc[i,'relative']='Protist'
            elif row.DIAMOND_domain=='Viruses' or row.DIAMOND_domain=='Bacteria':
                dfm.loc[i,'relative']=row.DIAMOND_domain
    elif str(row.tree_relative)!='nan':

        dfm.loc[i,'relative']=row.tree_relative


In [36]:
for index, row in dfm.iterrows():
    dfm.loc[index, 'Loc']=deep.loc[index,'Localizations']
    

In [46]:
dfm.loc[:,['Predicted function', 'CAZy family', 'Cell wall substrate ', 'source', 'relative', 'Loc']].to_csv("outputs/carbohydrate_catabolism.tsv",sep="\t")

In [30]:
##subset to those included in SI Figure 12
dfm=dfm[(dfm['Cell wall substrate '].astype(str)!='nan')|dfm['Predicted function'].isin(['Sodium-dependent glucose transporter (SGLT)', 'L-fucose-proton symporter'])]
for index, row in dfm.iterrows():
    c=row['CAZy family']
    a=row['Final annotation']
    if str(c)!='nan':
        dfm.loc[index,'Predicted function']=f'{a} ({c})'
        
len(list(dfm.index))

125

In [25]:
dfloc=pd.DataFrame()
for e, dfix in dfm.groupby('Final annotation'):
    dfloc.loc[e,'N_HTGs']=dfix.shape[0]
    dfloc.loc[e,'Substrate']=dfix['Cell wall substrate '].values[0]
    dfloc.loc[e,'N_Ex/M']=dfix[dfix.Loc.str.contains('Cell membrane')|dfix.Loc.str.contains('Extracellular')].shape[0]
    

In [1]:
import pandas as pd

In [112]:
# from collections import Counter

for s,ds in dfm.groupby('Cell wall substrate '):
    print(s)
    for g, dfi in ds.groupby('Final annotation'):
        print(g,Counter(dfi.relative))
    print(" ")

Cellulose
Endoglucanase (GH45) Counter({'Collembola': 5, 'Rotifera': 2})
Endoglucanase (GH9) Counter({'Collembola': 8, 'Branchiopoda': 1})
 
Chitin 
Chitin deacetylase  (CE4) Counter({'Collembola': 1})
Chitinase (GH18) Counter({'Fungi': 1, 'Viridiplantae': 1})
Chitinase (GH19) Counter({'Collembola': 8})
Endochitinase-like (GH?) Counter({'Fungi': 1})
Lytic chitin monooxygenase (AA15) Counter({'Collembola': 5})
N-acetyl-β-hexosaminidase (GH20) Counter({'Collembola': 3, 'Fungi': 1})
 
Mannan/hemicellulose
endo-1,4-β-mannosidase (GH5_10) Counter({'Collembola': 1})
β-mannosidase (GH130) Counter({'Collembola': 1})
β-mannosidase (GH2) Counter({'Collembola': 2})
 
Pectin 
Pectin methylesterase (CE8) Counter({'Collembola': 1, 'Bacteria': 1})
Pectin/pectate lyase (PL1_7) Counter({'Collembola': 6})
Pectin/pectate lyase-like (PL?) Counter({'Collembola': 1})
Polygalacturonase (GH28) Counter({'Collembola': 5})
 
Peptidoglycan
Cell wall hydrolase SleB Counter({'Collembola': 1})
Domesticated amidase e

## Fill in DNA repair ancestry

In [9]:
dna=pd.read_csv("outputs/dna_repair.tsv",sep="\t",index_col=0)

In [7]:
for m, row in dna.iterrows():
    dna.loc[m,['DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[m,['domain','kingdom','phylum','class','order']].values
for index, row in dna.iterrows():
    if index in dfs.index:
        dna.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values
for index, row in dna.iterrows():
    if index in set(dfriso[dfriso.tree_HTG].index):
        dna.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]
        dna.loc[index,'tree']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].index.values[0]

/tmp/ipykernel_3609942/1966138207.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Metazoa' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dna.loc[m,['DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[m,['domain','kingdom','phylum','class','order']].values
/tmp/ipykernel_3609942/1966138207.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Insecta' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dna.loc[m,['DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[m,['domain','kingdom','phylum','class','order']].values
/tmp/ipykernel_3609942/1966138207.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in 

In [14]:
from Bio import SeqIO
td=set(dna.index)-set(deep.index)
input_file = open(f"ncbi_dataset/data/GCF_014529535.2/protein.faa") 
fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
f=open('missing_deeploc_dna.fasta','w')
for x in td:
    f.write(f'>{x}\n')
    f.write(str(fasta[x].seq)+'\n')
f.close()


In [16]:
for i, row in dna.iterrows():
    if str(row.tree_relative)=='nan':
    
        if row.tree_phylum=='Arthropoda':
            if row.tree_class!='nan':
                dna.loc[i,'relative']=row.tree_class
        elif row.tree_kingdom=='Metazoa':
            if row.tree_phylum!='nan':
                dna.loc[i,'relative']=row.tree_phylum
        elif row.tree_kingdom=='Fungi':
            dna.loc[i,'relative']='Fungi'
        elif row.tree_kingdom=='Viridiplantae':
            dna.loc[i,'relative']='Viridiplantae'
        elif row.tree_domain=='Eukaryota' and str(row.tree_kingdom)!='nan':
            dna.loc[i,'relative']='Protist'
        elif row.tree_domain=='Viruses' or row.tree_domain=='Bacteria':
            dna.loc[i,'relative']=row.tree_domain
        if str(dna.loc[i,'relative'])=='nan':
            # print(row.DIAMOND_kingdom)
    
            if row.DIAMOND_phylum=='Arthropoda':
                if row.DIAMOND_class!='nan':
                    dna.loc[i,'relative']=row.DIAMOND_class
            elif row.DIAMOND_kingdom=='Metazoa':
                if row.DIAMOND_phylum!='nan':
                    dna.loc[i,'relative']=row.DIAMOND_phylum
            elif row.DIAMOND_kingdom=='Fungi':
                dna.loc[i,'relative']='Fungi'
            elif row['DIAMOND_kingdom']=='Viridiplantae':
                dna.loc[i,'relative']='Viridiplantae'
            elif row.DIAMOND_domain=='Eukaryota' and row.DIAMOND_kingdom!='nan':
                dna.loc[i,'relative']='Protist'
            elif row.DIAMOND_domain=='Viruses' or row.DIAMOND_domain=='Bacteria':
                dna.loc[i,'relative']=row.DIAMOND_domain
    elif str(row.tree_relative)!='nan':

        dna.loc[i,'relative']=row.tree_relative


In [27]:
for index, row in dna.iterrows():
    if 'Dipt' in row.Ancestry :
        dna.loc[index,'relative']='Dipteran'

In [33]:
dna.loc[:,['Annotation', 'Predicted function', 'relative']].to_csv("outputs/dna_repair.tsv",sep="\t")

## Kynurenine pathway ancestry

In [52]:
dna=pd.read_csv("outputs/tryptophan_kynurenine.tsv",sep="\t",index_col=0)

In [64]:
for m, row in dna.iterrows():
    dna.loc[m,['DIAMOND_domain','DIAMOND_kingdom','DIAMOND_phylum','DIAMOND_class','DIAMOND_order']]=dfriso.loc[m,['domain','kingdom','phylum','class','order']].values
for index, row in dna.iterrows():
    if index in dfs.index:
        dna.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values
for index, row in dna.iterrows():
    if index in set(dfriso[dfriso.tree_HTG].index):
        dna.loc[index,'tree_relative']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].values[0]
        dna.loc[index,'tree']=dft[dft['cluster_seqs'].str.contains(index)]['primary neighbor'].index.values[0]

/tmp/ipykernel_3609942/1966138207.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Mucoromycetes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dna.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values
/tmp/ipykernel_3609942/1966138207.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Mucorales' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dna.loc[index,['tree_domain','tree_kingdom','tree_phylum','tree_class','tree_order']]=dfs.loc[index,['domain','kingdom','phylum','class','order']].values


In [65]:
for i, row in dna.iterrows():
    if str(row.tree_relative)=='nan':
    
        if row.tree_phylum=='Arthropoda':
            if row.tree_class!='nan':
                dna.loc[i,'relative']=row.tree_class
        elif row.tree_kingdom=='Metazoa':
            if row.tree_phylum!='nan':
                dna.loc[i,'relative']=row.tree_phylum
        elif row.tree_kingdom=='Fungi':
            dna.loc[i,'relative']='Fungi'
        elif row.tree_kingdom=='Viridiplantae':
            dna.loc[i,'relative']='Viridiplantae'
        elif row.tree_domain=='Eukaryota' and str(row.tree_kingdom)!='nan':
            dna.loc[i,'relative']='Protist'
        elif row.tree_domain=='Viruses' or row.tree_domain=='Bacteria':
            dna.loc[i,'relative']=row.tree_domain
        if str(dna.loc[i,'relative'])=='nan':
            # print(row.DIAMOND_kingdom)
    
            if row.DIAMOND_phylum=='Arthropoda':
                if row.DIAMOND_class!='nan':
                    dna.loc[i,'relative']=row.DIAMOND_class
            elif row.DIAMOND_kingdom=='Metazoa':
                if row.DIAMOND_phylum!='nan':
                    dna.loc[i,'relative']=row.DIAMOND_phylum
            elif row.DIAMOND_kingdom=='Fungi':
                dna.loc[i,'relative']='Fungi'
            elif row['DIAMOND_kingdom']=='Viridiplantae':
                dna.loc[i,'relative']='Viridiplantae'
            elif row.DIAMOND_domain=='Eukaryota' and row.DIAMOND_kingdom!='nan':
                dna.loc[i,'relative']='Protist'
            elif row.DIAMOND_domain=='Viruses' or row.DIAMOND_domain=='Bacteria':
                dna.loc[i,'relative']=row.DIAMOND_domain
    elif str(row.tree_relative)!='nan':

        dna.loc[i,'relative']=row.tree_relative


In [67]:
for index, row in dna.iterrows():
    if 'Dipt' in row.Ancestry :
        dna.loc[index,'relative']='Dipteran'

In [69]:
dna.loc[:,['Annotation', 'Predicted function', 'EC number', 'relative']].to_csv("outputs/tryptophan_kynurenine.tsv",sep="\t")